In [ ]:
# ==========================================
# EE214    Assignment 2    Clustering Skeleton
# ==========================================
# Code skeleton for Task 2. You may change it as you deem necessary.

#---------- 0. Imports---------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Install required UCI repo fetcher (only once)
try:
    from ucimlrepo import fetch_ucirepo     # one line UCI fetch
except ImportError:
    !pip install ucimlrepo
    from ucimlrepo import fetch_ucirepo

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

# For AutoEncoder
import torch, torch.nn as nn, torch.optim as optim

# Fix seed for reproducibility
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
#---------- 1. Data Loading---------
#-----1.1 Dermatology dataset-------
import os

filename = "dermatology_dataset.csv"

if os.path.exists(filename):
    print(f"Loading {filename} from local.")
    X_raw_df = pd.read_csv(filename)
else:
    print(f"Fetching {filename} from UCI repository.")
    derm = fetch_ucirepo(name="Dermatology")       # TODO: cache locally if slow
    X_raw_df = derm.data.features                  # 34 features (no labels)
    X_raw_df.to_csv(filename, index=False)
    print(f"Saved {filename} to local.")

# Drop the samples with missing values
X_raw_df.replace(to_replace="?", value=np.nan, inplace=True)
X_raw_df.dropna(inplace=True)
X_raw_df = X_raw_df.astype(float)
X_raw = X_raw_df.to_numpy()

#-----1.2 Unknown dataset----------
print("------------------------------")
X_unknown_df = pd.read_csv("unknown_dataset.csv")   # 5452 10 PCA
print("Check whether there is missing value in \"unknown_dataset.csv\"")
X_unknown_df.replace("?", np.nan, inplace=True)
print(X_unknown_df.isnull().sum())
X_unknown_df.dropna(inplace=True)
X_raw_df = X_raw_df.astype(float)
X_unknown = X_unknown_df.to_numpy()

print("------------------------------")
print(f"X_raw shape: {X_raw.shape}")            # (358, 34)
print(f"X_unknown shape: {X_unknown.shape}")    # (5452, 10)

In [ ]:
#----------2. Scaling---------
# Standard Scaler
std_scaler = StandardScaler()
X_std = std_scaler.fit_transform(X_raw)

# MinMax Scaler
minmax_scaler = MinMaxScaler()                  # for comparison
X_minmax = minmax_scaler.fit_transform(X_raw)

# Compare the results between scaled data by standard and minmax scalers (Visualization)
plt.figure(figsize=(7, 3))
plt.plot(X_std[0], label="StandardScaler")
plt.plot(X_minmax[0], label="MinMaxScaler")
plt.title("Values of Scaled First Sample")
plt.xlabel("Feature Index")
plt.ylabel("Scaled Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#----------3. Dimensionality Reduction---------
# PCA which can explain 95% of the variance.
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_std)

print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")
print(f"The number of components selected: {pca.n_components_}")
print(f"X_pca shape: {X_pca.shape}")
print("------------------------------")

#---AutoEncoder stub------------------------------------------------
class AE(nn.Module):
    def __init__(self, in_dim, latent_dim=10):
        super().__init__()
        # TODO define encoder / decoder layers
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 24),
            nn.ReLU(),
            nn.Linear(24, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 24),
            nn.ReLU(),
            nn.Linear(24, in_dim)
        )

    def forward(self, x):
        # TODO forward pass
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z   # return output (x_recon) and latent vector (z)

def train_ae(x, latent_dim=10, epochs=100, lr=1e-3, seed=42):
    """Returns latent representation (np.ndarray)."""
    # TODO: build model, optimizer, loss, training loop
    set_seed(seed)

    model = AE(x.shape[1], latent_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # Training
    model.train()
    losses = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        x_recon, _ = model(x)
        loss = criterion(x_recon, x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch + 1:4d}, Loss: {loss.item():.4f}")

    # Generating latent feature
    model.eval()
    with torch.no_grad():
        _, latent = model(x)
        x_np = latent.detach().cpu().numpy()

    return x_np, losses

# Convert numpy (X_std) to tensor (X_tensor)
X_tensor = torch.tensor(X_std, dtype=torch.float32)

latent_dim = 10
epochs = 2500
print(f"Training AE with latent dim: {latent_dim}")
X_ae, losses = train_ae(X_tensor, latent_dim, epochs)

print("------------------------------")
print(f"X_ae shape: {X_ae.shape}")

# Visualize the graph Loss vs. Epoch
plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.title(f"Loss vs. Epoch with latent dim {latent_dim}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.show()
#---------------------------------------------------------------------

In [ ]:
# Features from Dermatology dataset
features_derm = [X_std, X_pca, X_ae]
names_derm = ["raw", "PCA", "AE"]

In [ ]:
#----------4. Helper: k Means Elbow & Silhouette---------
def elbow_silhouette_plot(features, names, k_range=range(2, 16), random_state=42):
    fig, axes = plt.subplots(len(names), 2, figsize=(9, 3*len(names)))

    # for each corresponding pair of 'features' and 'names'
    for i, (X, name) in enumerate(zip(features, names)):
        inertia, sil = [], []

        for k in k_range:
            # TODO : run KMeans, get inertia and Silhouette scores
            model = KMeans(n_clusters=k, random_state=random_state, n_init="auto")
            model.fit(X)
            inertia.append(model.inertia_)
            sil.append(silhouette_score(X, model.labels_))


        # TODO: select best k by inspecting the plots
        if len(names) == 1:
            axes = [axes]
        ax0, ax1 = axes[i]

        # Elbow Plot
        ax0.plot(k_range, inertia, 'bo-')
        ax0.set_title(f"{name} - Elbow")
        ax0.set_xlabel("k")
        ax0.set_ylabel("Inertia")
        ax0.grid(True)

        # Silhouette Plot
        best_k = np.argmax(sil) + k_range.start
        best_score = np.max(sil)
        ax1.plot(k_range, sil, 'ro-')
        ax1.set_title(f"{name} - Silhouette\nbest_k = {best_k} / best_score = {best_score:.4f}")
        ax1.set_xlabel("k")
        ax1.set_ylabel("Silhouette Score")
        ax1.grid(True)

    plt.tight_layout()
    plt.show()

elbow_silhouette_plot(features_derm, names_derm)

In [ ]:
#----------5. Helper: Agglomerative dendrogram---------
from scipy.cluster.hierarchy import linkage, dendrogram
def dendro_plot(features, names, methods=["ward", "average", "complete"], k_range=range(2, 16)): # TODO : use other linkage methods for comparison
    fig, axes = plt.subplots(2*len(names), 3, figsize=(20, 6*len(names)))

    for i, (X, name) in enumerate(zip(features, names)):
        for j, method in enumerate(methods):
            # Row for dendrogram: 2*i
            # Row for silhouette: 2*i+1
            ax_dend = axes[2*i][j]
            ax_sil = axes[2*i+1][j]

            # TODO : plot dendogram
            Z = linkage(X, method=method)   # TODO : get linkage
            dendrogram(Z, ax=ax_dend)
            ax_dend.set_title(f"{name} - Dendrogram ({method})")
            ax_dend.set_xlabel("Samples")
            ax_dend.set_ylabel("Distance")

            # TODO: compute Silhouette for k=2..15
            sil = []
            for k in k_range:
                model = AgglomerativeClustering(n_clusters=k, linkage=method)
                model.fit(X)
                sil.append(silhouette_score(X, model.labels_))

            # TODO: select best k by inspecting the plot
            best_k = np.argmax(sil) + k_range.start
            best_score = np.max(sil)
            ax_sil.plot(k_range, sil, 'ro-')
            ax_sil.set_title(f"{name} - Silhouette ({method})\nbest_k = {best_k} / best_score = {best_score:.4f}")
            ax_sil.set_xlabel("k")
            ax_sil.set_ylabel("Silhouette Score")
            ax_sil.grid(True)

    plt.tight_layout()
    plt.show()

dendro_plot(features_derm, names_derm)

In [ ]:
#----------6. Helper: DBSCAN grid heatmap---------
def dbscan_heatmap(features, names, eps_vals, min_samples_vals):
    """
    Highlevel outline:
        1. Iterate over every (eps, min_samples) combination.
        2. Fit DBSCAN, record:
            number of clusters (ignore noise label -1)
            Silhouette score (set to NaN if clusters < 2)
        3. Pivot the results into two matrices:
            heatmap of cluster counts
            heatmap of Silhouette scores
        4. Display the two heatmaps side byside .
        5. After inspecting the visuals, manually choose the
            best (eps, min_samples) pair for your final model.
    """
    # TODO: implement the loop, DataFrame creation, seaborn heatmaps,
    # and return or log the chosen parameters.
    dfs = []

    fig, axes = plt.subplots(len(names), 3, figsize=(15, 5*len(names)))

    for i, (X, name) in enumerate(zip(features, names)):
        results = []

        for eps in eps_vals:
            for min_samples in min_samples_vals:
                model = DBSCAN(eps=eps, min_samples=min_samples)
                model.fit(X)
                labels = model.labels_
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

                # Compute noise ratio
                noise_ratio = np.sum(labels == -1) / len(labels)

                if n_clusters < 2:
                    sil = np.nan
                else:
                    mask = labels != -1
                    sil = silhouette_score(X[mask], labels[mask])

                # Gather the results of computing
                results.append({
                    "eps": np.round(eps, 2),
                    "min_samples": min_samples,
                    "n_clusters": n_clusters,
                    "sil": sil,
                    "noise_ratio": noise_ratio
                })

        df = pd.DataFrame(results)
        dfs.append(df)
        pivot_n_clusters = df.pivot(index="eps", columns="min_samples", values="n_clusters").sort_index(ascending=False)
        pivot_sil = df.pivot(index="eps", columns="min_samples", values="sil").sort_index(ascending=False)
        pivot_noise = df.pivot(index="eps", columns="min_samples", values="noise_ratio").sort_index(ascending=False)

        if len(names) == 1:
            axes = [axes]
        ax0, ax1, ax2 = axes[i]

        # Plot heatmap of the number of clusters
        sns.heatmap(pivot_n_clusters, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax0)
        ax0.set_title(f"{name} - Number of Clusters")
        ax0.set_xlabel("min_samples")
        ax0.set_ylabel("eps")

        # Plot heatmap of silhouette score
        sns.heatmap(pivot_sil, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax1)
        ax1.set_title(f"{name} - Silhouette Score")
        ax1.set_xlabel("min_samples")
        ax1.set_ylabel("eps")

        # Plot heapmap of noise ratio
        sns.heatmap(pivot_noise, annot=True, fmt=".2f", cmap="coolwarm", ax=ax2)
        ax2.set_title(f"{name} - Noise Ratio")
        ax2.set_xlabel("min_samples")
        ax2.set_ylabel("eps")

    plt.tight_layout()
    plt.show()

    return dfs

eps_vals = np.round(np.linspace(1.0, 5.0, 21), 2)
min_samples_vals = range(2, 12)

dfs = dbscan_heatmap(features_derm, names_derm, eps_vals, min_samples_vals)

In [ ]:
#----------7. Final clustering (studentchosen params)---------
# TODO: run k Means , Agglomerative, DBSCAN with the parameters you selected
labels_km = KMeans(n_clusters=4, random_state=42, n_init="auto").fit_predict(X_ae)
labels_agg = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(X_ae)
labels_db = DBSCAN(eps=3.8, min_samples=4).fit_predict(X_ae)

# Print all classes labels
print(f"KMeans:         {np.unique(labels_km)}")
print(f"Agglomerative:  {np.unique(labels_agg)}")
print(f"DBSCAN:         {np.unique(labels_db)} (includes -1 if noise)")

In [ ]:
#----------8. Visualisation helper---------
def scatter_2d(X2, labels, title):
    plt.figure(figsize=(4,4))

    # Scatter plot with labels
    unique_labels = np.unique(labels)
    for label in unique_labels:
        mask = labels == label
        label_name = f"Cluster {label}" if label != -1 else "Noise"
        plt.scatter(X2[mask, 0], X2[mask, 1], label=label_name, s=8)

    plt.legend()
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Example: scatter_2d(X_pca, labels_km, "Dermatology: k Means on PCA")
pca2 = PCA(n_components=2, random_state=42)
X_ae2 = pca2.fit_transform(X_ae)

scatter_2d(X_ae2, labels_km, "Dermatology: k Means on AE")
scatter_2d(X_ae2, labels_agg, "Dermatology: Agglomerative on AE")
scatter_2d(X_ae2, labels_db, "Dermatology: DBSCAN on AE")

In [ ]:
#----------9. UNKNOWN DATASET---------
# Repeat steps 4 8 for X_unknown (skip AE training).
features_unknown = [X_unknown]
names_unknown = ["PCA"]

In [ ]:
elbow_silhouette_plot(features_unknown, names_unknown)

In [ ]:
dendro_plot(features_unknown, names_unknown)

In [ ]:
eps_vals_unknown = np.round(np.linspace(0.5, 2.0, 10), 2)
min_samples_vals_unknown = range(15, 25)

dfs_unknown = dbscan_heatmap(features_unknown, names_unknown, eps_vals_unknown, min_samples_vals_unknown)

In [ ]:
labels_km_unknown = KMeans(n_clusters=4, random_state=42, n_init="auto").fit_predict(X_unknown)
labels_agg_unknown = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(X_unknown)
labels_db_unknown = DBSCAN(eps=1.33, min_samples=19).fit_predict(X_unknown)

print(f"KMeans:         {np.unique(labels_km_unknown)}")
print(f"Agglomerative:  {np.unique(labels_agg_unknown)}")
print(f"DBSCAN:         {np.unique(labels_db_unknown)} (includes -1 if noise)")

In [ ]:
scatter_2d(X_unknown, labels_km_unknown, "Unknown: k Means on PCA")
scatter_2d(X_unknown, labels_agg_unknown, "Unknown: Agglomerative on PCA")
scatter_2d(X_unknown, labels_db_unknown, "Unknown: DBSCAN on PCA")

In [ ]:
# Appendix 1 - Representation Effect (raw, PCA, AE)
labels_km_raw = KMeans(n_clusters=5, random_state=42, n_init="auto").fit_predict(X_std)
labels_km_pca = KMeans(n_clusters=5, random_state=42, n_init="auto").fit_predict(X_pca)

X_std2 = pca2.fit_transform(X_std)

scatter_2d(X_std2, labels_km_raw, "Dermatology: k Means on raw")
scatter_2d(X_pca, labels_km_pca, "Dermatology: k Means on PCA")

In [ ]:
# Appendix 2 - Stability Assessment
for i in range(1, 5):
    labels_km_i = KMeans(n_clusters=4, random_state=i, n_init="auto").fit_predict(X_ae)
    scatter_2d(X_ae2, labels_km_i, f"Dermatology: k-Means Clustering on AE with seed {i}")

In [ ]:
# Appendix 3 - Alternative Approaches or Improvements
#----------DBSCAN grid heatmap with Davies–Bouldin Index---------
from sklearn.metrics import davies_bouldin_score

def dbscan_heatmap_dbi(features, names, eps_vals, min_samples_vals):
    dfs = []

    fig, axes = plt.subplots(len(names), 3, figsize=(15, 5*len(names)))

    for i, (X, name) in enumerate(zip(features, names)):
        results = []

        for eps in eps_vals:
            for min_samples in min_samples_vals:
                model = DBSCAN(eps=eps, min_samples=min_samples)
                model.fit(X)
                labels = model.labels_
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

                # Compute noise ratio
                noise_ratio = np.sum(labels == -1) / len(labels)

                # Compute DBI instead of Silhouette score
                if n_clusters < 2:
                    dbi = np.nan
                else:
                    mask = labels != -1
                    dbi = davies_bouldin_score(X[mask], labels[mask])

                results.append({
                    "eps": np.round(eps, 2),
                    "min_samples": min_samples,
                    "n_clusters": n_clusters,
                    "dbi": dbi,
                    "noise_ratio": noise_ratio
                })

        df = pd.DataFrame(results)
        dfs.append(df)
        pivot_n_clusters = df.pivot(index="eps", columns="min_samples", values="n_clusters").sort_index(ascending=False)
        pivot_dbi = df.pivot(index="eps", columns="min_samples", values="dbi").sort_index(ascending=False)
        pivot_noise = df.pivot(index="eps", columns="min_samples", values="noise_ratio").sort_index(ascending=False)

        if len(names) == 1:
            axes = [axes]
        ax0, ax1, ax2 = axes[i]

        # Plot heatmap of the number of clusters
        sns.heatmap(pivot_n_clusters, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax0)
        ax0.set_title(f"{name} - Number of Clusters")
        ax0.set_xlabel("min_samples")
        ax0.set_ylabel("eps")

        # Plot heatmap of Davies–Bouldin index
        sns.heatmap(pivot_dbi, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax1)
        ax1.set_title(f"{name} - Davies–Bouldin Index")
        ax1.set_xlabel("min_samples")
        ax1.set_ylabel("eps")

        # Plot heapmap of noise rate
        sns.heatmap(pivot_noise, annot=True, fmt=".2f", cmap="coolwarm", ax=ax2)
        ax2.set_title(f"{name} - Noise Ratio")
        ax2.set_xlabel("min_samples")
        ax2.set_ylabel("eps")

    plt.tight_layout()
    plt.show()

    return dfs

eps_vals = np.round(np.linspace(1.0, 5.0, 21), 2)
min_samples_vals = range(2, 12)

dfs_dbi = dbscan_heatmap_dbi([X_ae], ["AE"], eps_vals, min_samples_vals)

labels_db_dbi = DBSCAN(eps=4.0, min_samples=5).fit_predict(X_ae)

scatter_2d(X_ae2, labels_db_dbi, "Dermatology: DBSCAN on AE using DBI")

In [ ]:
# Appendix 4 - Comparison k-Means Clusterings using X_std and X_minmax
X_minmax_pca = pca.fit_transform(X_minmax)

# Convert numpy (X_minmax) to tensor (X_minmax_tensor)
X_minmax_tensor = torch.tensor(X_minmax, dtype=torch.float32)
X_minmax_ae, losses = train_ae(X_minmax_tensor, latent_dim, epochs)

elbow_silhouette_plot([X_minmax, X_minmax_pca, X_minmax_ae], ["minmax", "minmax_pca", "minmax_ae"])

labels_km_minmax = KMeans(n_clusters=3, random_state=42, n_init="auto").fit_predict(X_minmax)
labels_km_minmax_pca = KMeans(n_clusters=3, random_state=42, n_init="auto").fit_predict(X_minmax_pca)
labels_km_minmax_ae = KMeans(n_clusters=3, random_state=42, n_init="auto").fit_predict(X_minmax_ae)

X_minmax2 = pca2.fit_transform(X_minmax)
X_minmax_ae2 = pca2.fit_transform(X_minmax_ae)

scatter_2d(X_minmax2, labels_km_minmax, "Dermatology: k Means on minmax")
scatter_2d(X_minmax_pca, labels_km_minmax_pca, "Dermatology: k Means on minmax_pca")
scatter_2d(X_minmax_ae2, labels_km_minmax_ae, "Dermatology: k Means on minmax_ae")